In [0]:
from pyspark.sql import functions as F

silver = spark.table("telecom.silver.sip_messages")

# Only REGISTER transactions (requests + their responses)
reg = silver.filter(F.col("cseq_method") == "REGISTER")

# 1. One row per registration attempt (Call-ID + CSeq number)
attempts = (reg.groupBy("call_id", "cseq_num")
    .agg(
        F.min(F.when(F.col("method") == "REGISTER", F.col("event_time_utc"))).alias("request_time"),
        F.max(F.when(F.col("status_code") >= 200, F.col("status_code"))).alias("final_status"),
        F.min(F.when(F.col("status_code") >= 200, F.col("event_time_utc"))).alias("final_response_time"),
        F.first("source_file").alias("source_file"),
    )
    .withColumn("response_ms",
        (F.col("final_response_time").cast("double") - F.col("request_time").cast("double")) * 1000)
    .withColumn("outcome",
        F.when(F.col("final_status").between(200, 299), "SUCCESS")
         .when(F.col("final_status") == 401, "AUTH_CHALLENGE")
         .when(F.col("final_status").isNull(), "NO_RESPONSE")
         .otherwise("FAILURE"))
)

attempts.write.mode("overwrite").saveAsTable("telecom.gold.sip_registration_attempts")

# 2. Overall KPI summary
kpis = attempts.agg(
    F.count("*").alias("total_attempts"),
    F.sum(F.when(F.col("outcome") == "SUCCESS", 1).otherwise(0)).alias("successful"),
    F.sum(F.when(F.col("outcome") == "FAILURE", 1).otherwise(0)).alias("failed"),
    F.round(F.avg("response_ms"), 1).alias("avg_response_ms"),
).withColumn("success_rate_pct",
    F.round(F.col("successful") / F.col("total_attempts") * 100, 1))

kpis.write.mode("overwrite").saveAsTable("telecom.gold.sip_registration_kpis")

display(spark.table("telecom.gold.sip_registration_attempts"))
display(spark.table("telecom.gold.sip_registration_kpis"))